In [1]:
import numpy as np
import pandas as pd
from paraphrase_detection import data_shuffle_split, PairClassifier, Train, TextPairDataset, log_metrics_and_model, BOSearchTrain
import torch
from sentence_transformers import SentenceTransformer
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
from loguru import logger
import os
from transformers import AutoTokenizer
from enums import Optimizer, Scheduler

os.environ["TOKENIZERS_PARALLELISM"] = "false"

/Users/oli.dmrs/Documents/Personal Projects/paraphrase-detection-ml/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import numpy as np
import pandas as pd
from paraphrase_detection import data_shuffle_split
df = pd.read_parquet("hf://datasets/sentence-transformers/quora-duplicates/pair-class/train-00000-of-00001.parquet")
data = df.to_numpy()
data = data[:10000]

X = data[:, :2]
y = data[:, 2:3]

X_train, X_val, X_test, y_train, y_val, y_test = data_shuffle_split(X, y, 0.15, 0.12, 42)

### Bayesian Optimization search for CrossEntropy model

In [ ]:
sbert_model = SentenceTransformer("all-MiniLM-L12-v2")

search_params = ['lr', 'weight_decay', 'dropout_p', 'batch_size', 'n_freeze', 'use_n_layers', 'fc1', 'fc2', 'fc3']

bounds = torch.tensor([
    [1e-5, 0.001, 0.0, 16.0, 9.0, 1.0, 64.0, 64.0, 16.0],
    [1e-1, 0.1, 0.6, 512.0, sbert_model[0].auto_model.config.num_hidden_layers, 3.0, 256.0, 128.0, 64.0]
])

seed = 42
criterion = torch.nn.CrossEntropyLoss()
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))

bo_searcher = BOSearchTrain(
    bounds = bounds,
    names = search_params,
    sbert_model = sbert_model,
    X_train = X_train,
    y_train = y_train,
    X_val = X_val,
    y_val = y_val,
    optimizer = Optimizer.ADAMW,
    scheduler = Scheduler.LINEAR,
    criterion = criterion,
    device = device,
    n_init_samples = 10,
    n_iterations = 30,
    fixed = False,
    cos_similarity = False,
    epochs = 10,
    patience = 3
)

bo_searcher.search_loop()

2025-11-22 14:03:20.292 | INFO     | paraphrase_detection.hyperparameterselection:search_loop:317 - Starting BO hyperparameter search 

2025-11-22 14:03:20.292 | INFO     | paraphrase_detection.hyperparameterselection:init_fit_samples:146 - Initial fit of GP for 10 samples of hyperparameter sets 

2025-11-22 14:03:20.294 | INFO     | paraphrase_detection.hyperparameterselection:init_fit_samples:155 - Starting iteration 0:
[('lr', 0.001092419377528131), ('weight_decay', 0.05480135977268219), ('dropout_p', 0.4394954741001129), ('batch_size', 87.43457794189453), ('n_freeze', 9.172524452209473), ('use_n_layers', 1.3317302465438843), ('fc1', 76.52411651611328), ('fc2', 118.35523986816406), ('fc3', 37.08708953857422)]


EPOCH: 0


2025-11-22 14:07:33.791 | INFO     | paraphrase_detection.train:run_training_loop:171 - Epoch 0: train loss = 0.5306032130191493
2025-11-22 14:07:33.792 | INFO     | paraphrase_detection.train:run_training_loop:172 - Epoch 0: validation loss = 0.45354922115802765
2025-11-22 14:07:33.793 | INFO     | paraphrase_detection.train:run_training_loop:173 - Epoch 0: train acc = 0.7272727272727273
2025-11-22 14:07:33.793 | INFO     | paraphrase_detection.train:run_training_loop:174 - Epoch 0: validation acc = 0.7823529411764706
2025-11-22 14:07:33.794 | INFO     | paraphrase_detection.train:run_training_loop:175 - Epoch 0: validation f1 = 0.779298245614035


EPOCH: 1


In [4]:
print(len(search_params))

10


In [3]:
seed = 42
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
fc_layer_sizes = [sbert_model.get_sentence_embedding_dimension(), sbert_model.get_sentence_embedding_dimension() // 2]
dropout_p = 0.1
lr = 1e-3
epochs = 10
batch_size = 32
steps = epochs * X_train.shape[0] // batch_size
n_warmup_steps = steps * 0.1
n_freeze = 3
weight_decay = 0.01

np.random.seed(seed)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
model = SBERTPairClassifier(sbert_model, fc_layer_sizes, device, True, dropout_p)
optimizer = torch.optim.AdamW(model.parameters(), lr, weight_decay = weight_decay)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps = n_warmup_steps, num_training_steps = steps)
criterion = torch.nn.CrossEntropyLoss()

ModuleList(
  (0): Linear(in_features=1152, out_features=384, bias=True)
  (1): Linear(in_features=384, out_features=192, bias=True)
  (2): Linear(in_features=192, out_features=2, bias=True)
)


In [ ]:
train_loader = DataLoader(dataset = TextPairDataset(X_train, y_train), batch_size = batch_size, shuffle = True, num_workers = 4)
validation_loader = DataLoader(dataset = TextPairDataset(X_val, y_val), batch_size = batch_size, shuffle = True, num_workers = 4)
test_loader = DataLoader(dataset = TextPairDataset(X_test, y_test), batch_size = batch_size, shuffle = True, num_workers = 4)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenized_train_loader = DataLoader(dataset = TextPairDataset(X_train, y_train, tokenization=True, tokenizer=tokenizer), batch_size = batch_size, shuffle = True, num_workers = 4)
tokenized_validation_loader = DataLoader(dataset = TextPairDataset(X_val, y_val, tokenization=True, tokenizer=tokenizer), batch_size = batch_size, shuffle = True, num_workers = 4)
tokenized_test_loader = DataLoader(dataset = TextPairDataset(X_test, y_test, tokenization=True, tokenizer=tokenizer), batch_size = batch_size, shuffle = True, num_workers = 4)

In [ ]:
#Fixed sbert model
train = Train(model = model,
              optimizer = optimizer,
              scheduler = scheduler,
              criterion = criterion, 
              device = device,
              n_freeze = n_freeze,
              epochs = epochs,
              train_dataloader = tokenized_train_loader,
              val_dataloader = tokenized_test_loader,
              sbert_trainable = True
              )

best_params, avg_batch_train_loss, epoch_train_acc, avg_batch_val_loss, epoch_val_acc, epoch_val_f1 = train.run_training_loop()

False
False
False
False
False
EPOCH: 0


2025-11-20 18:34:35.229 | INFO     | paraphrase_detection.train:run_training_loop:126 - Epoch 0: train loss = 0.5515606858027287
2025-11-20 18:34:35.231 | INFO     | paraphrase_detection.train:run_training_loop:127 - Epoch 0: validation loss = 0.6087496508943274
2025-11-20 18:34:35.231 | INFO     | paraphrase_detection.train:run_training_loop:128 - Epoch 0: train acc = 0.707620320855615
2025-11-20 18:34:35.232 | INFO     | paraphrase_detection.train:run_training_loop:129 - Epoch 0: validation acc = 0.642
2025-11-20 18:34:35.232 | INFO     | paraphrase_detection.train:run_training_loop:130 - Epoch 0: validation f1 = 0.6419807465201461


EPOCH: 1


In [6]:
log_metrics_and_model('SBERTv1', avg_batch_train_loss, epoch_train_acc, avg_batch_val_loss, \
                      epoch_val_acc, epoch_val_f1, best_params)